## Initial Setup

In [ ]:
import pandas as pd

annotations_path = '/Users/brukewossenseged/Desktop/projects/sails/RMM_with_ELE_Highlighted.xlsx'

# Load the first Excel sheet using openpyxl engine
bw_df = pd.read_excel(annotations_path, engine='openpyxl')

In [ ]:
unq_ids = bw_df["ID"][bw_df["SourceFile"].str[0].isin({"A", "B", "C", "D"})].dropna().unique()
unq_ids = bw_df["ID"].dropna().unique()
print(len(unq_ids))
print(' '.join(unq_ids))

In [ ]:
# Get all columns that match RMM_X pattern (where X is a number)
rmm_cols = [col for col in bw_df.columns if col.startswith('RMM_') and 'TS' not in col and "Notes" not in col]

# Get all unique values from RMM_X columns
unique_rmm_values = set()

for col in rmm_cols:
    # Get unique non-null values from this column
    unique_values = bw_df[col].dropna().unique()
    unique_rmm_values.update(val.strip() for val in unique_values if isinstance(val, str))

print("Unique RMM values:")
print(sorted(unique_rmm_values))

In [ ]:
# get column index of rmm_cols in df
rmm_cols = [(col, bw_df.columns.get_loc(col), bw_df.columns.get_loc(f"{col}_TS")) for col in rmm_cols]
print(rmm_cols)

### Extract Cell Colors from excel

In [ ]:
from openpyxl import load_workbook

COLOR_TO_LABEL = {
    12: "Unsure",
    13: "Not_Repetitive",
    14: "False_in_Context",
    16: "False_in_Context",
    15: "Not_Repetitive",
    17: "Debatable",
    18: "Video_Issue",
    19: "Many_Occurrences",
}

RMM_COLS = [col[1] for col in rmm_cols]  # Get column indices for RMM columns

# Load workbook with openpyxl to access formatting
wb = load_workbook(annotations_path)
ws = wb.active

# Create a dictionary to store colors by row
row_colors = {}

for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
    # Check the first cell (column A) for highlighting
    for ix, col in enumerate(RMM_COLS):
        cell = row[col]
        rmm_num = ix + 1

        row_colors.setdefault(row_idx, {})
        
        if cell.fill.patternType == 'solid':
            # Handle different color types
            fg_color = cell.fill.fgColor
            
            if fg_color.type == 'rgb' and fg_color.rgb:
                row_colors[row_idx][rmm_num] = fg_color.rgb
            elif fg_color.type == 'indexed' and fg_color.indexed:
                row_colors[row_idx][rmm_num] = f"{fg_color.indexed}"
            elif fg_color.type == 'theme' and fg_color.theme is not None:
                row_colors[row_idx][rmm_num] = f"{fg_color.theme}"

print(row_colors)

bw_df['RMM_1_Label'] = None
bw_df['RMM_2_Label'] = None
bw_df['RMM_3_Label'] = None
bw_df['RMM_4_Label'] = None
bw_df['RMM_5_Label'] = None

for idx in row_colors:
    df_idx = idx - 2  # Convert Excel row to DataFrame index (subtract header row)
    if df_idx < len(bw_df):
        for rmm_num, color in row_colors[idx].items():
            label = COLOR_TO_LABEL.get(int(color), None)
            if label:
                bw_df.at[df_idx, f'RMM_{rmm_num}_Label'] = label
    # if df_idx < len(bw_df):
    #     bw_df.at[df_idx, 'highlight_color'] = row_colors[idx]

bw_df.head()

In [ ]:
colors = set()
for row in row_colors:
    for col in row_colors[row]:
        color = row_colors[row][col]
        if color not in colors:
            print(row, row_colors[row])
        colors.add(row_colors[row][col])
print(colors)



# 2 {1: '9', 2: '9', 3: '9', 4: '9', 5: '9'}
# 19 {1: '12', 2: '12', 3: '12', 4: '12', 5: '12'}
# 25 {1: '13', 2: '9', 3: '9', 4: '9', 5: '9'}
# 62 {1: '14', 2: '14', 3: '14', 4: '14', 5: '14'}
# 82 {1: '15', 2: '15', 3: '15', 4: '15', 5: '15'}
# 89 {1: '16'}
# 121 {1: '17', 2: '17', 3: '17', 4: '17', 5: '17'}
# 136 {1: '18', 2: '18', 3: '18', 4: '18', 5: '18'}
# 200 {1: '19', 2: '19', 3: '9', 4: '9', 5: '9'}
# {'13', '17', '19', '18', '15', '9', '14', '16', '12'}

In [ ]:
rmm_cols_names = [col for col in bw_df.columns if col.startswith('RMM_') and 'TS' not in col and "Notes" not in col and "Label" not in col]
rmm_cols_labels = [col for col in bw_df.columns if col.startswith('RMM_') and "Label" in col]
rmm_cols_ts = [col for col in bw_df.columns if col.startswith('RMM_') and "TS" in col]
print(rmm_cols_labels)
print(rmm_cols_ts)
print(rmm_cols_names)
rmm_cols = list(zip(rmm_cols_names, rmm_cols_labels, rmm_cols_ts))

print(rmm_cols)
skip_labels = {'Unsure', 'Debatable'}

RMM_DUR_COUNT = {}
RMM_SEG_COUNT = {}
ID_TP_TO_SEG = {}

def parse_timestamps(ts_string):
    """Parse timestamp string with quality markers into list of (start, end, quality) tuples"""
    import re
    
    # Pattern to match timestamps with optional quality: MM:SS-MM:SS [quality] or MM.SS-MM.SS [quality]
    # This matches each timestamp segment individually
    pattern = re.compile(r'([\d:.]+)\s*-\s*([\d:.]+)(?:\s*\[(\d+)\])?')
    
    results = []
    for match in pattern.finditer(ts_string):
        start_str = match.group(1).replace('.', ':')  # Normalize to MM:SS format
        end_str = match.group(2).replace('.', ':')
        quality = int(match.group(3)) if match.group(3) else None
        
        # Convert to seconds
        start_sec = sum(float(x) * 60 ** i for i, x in enumerate(reversed(start_str.split(":"))))
        end_sec = sum(float(x) * 60 ** i for i, x in enumerate(reversed(end_str.split(":"))))
        
        results.append((start_sec, end_sec, quality))
    
    return results

for row in bw_df.iterrows():
    row_idx, row_data = row
    id, tp = row_data["ID"], row_data["timepoint"]
    ID_TP_TO_SEG.setdefault(id, {}).setdefault(tp, [])
    for col in rmm_cols:
        if pd.isna(row_data[col[0]]):
            continue
        if row_data[col[1]] in skip_labels:
            continue
        
        ts_string = str(row_data[col[2]]).strip()
        parsed_timestamps = parse_timestamps(ts_string)

        RMM_DUR_COUNT.setdefault(row_data[col[0]], 0)
        for ts in parsed_timestamps:
            RMM_DUR_COUNT[row_data[col[0]]] += ts[1] - ts[0]
        RMM_SEG_COUNT.setdefault(row_data[col[0]], 0)
        RMM_SEG_COUNT[row_data[col[0]]] += len(parsed_timestamps)
        print(f"Row {row_idx}, Column {col[0]}: {row_data[col[0]]}")
        print(f"  Label: {row_data[col[1]]}, Timestamps: {parsed_timestamps}")

In [ ]:
print(RMM_DUR_COUNT)
bw_rmm_map = {
    'arm flapping': 'hands flapping',
    'arms flapping': 'hands flapping',
    'one arm flap': 'one hand flap',
    'one arm flapping': 'one hand flap',
    'bouncing': 'jumping',
    'twisting': 'rocking',
    'bed jumping': 'jumping',
    'knee jumping': 'jumping',
}

def standardized(count: dict) -> dict:
    """Map RMM names to standardized names using bw_rmm_map"""
    NEW_COUNT = {}
    for rmm in count.keys():
        if rmm in bw_rmm_map:
            mapped_rmm = bw_rmm_map[rmm]
            NEW_COUNT.setdefault(mapped_rmm, 0)
            NEW_COUNT[mapped_rmm] += count[rmm]
            continue
        NEW_COUNT.setdefault(rmm, 0)
        NEW_COUNT[rmm] += count[rmm]
    return NEW_COUNT

RMM_DUR_COUNT_ST = standardized(RMM_DUR_COUNT)
RMM_SEG_COUNT_ST = standardized(RMM_SEG_COUNT)

print(RMM_DUR_COUNT_ST)
print(RMM_SEG_COUNT_ST)

In [ ]:
# Check dataframe columns and structure
print("Columns:", bw_df.columns.tolist())
print("\nDataframe shape:", bw_df.shape)
print("\nSample row:")
print(bw_df.iloc[0])

In [ ]:
main_RMM = {"hands flapping", "one hand flap", "jumping", "rocking", "spinning"} #,"clapping"}
for rmm in main_RMM:
    if rmm in RMM_SEG_COUNT_ST:
        print(f"{rmm}: {RMM_SEG_COUNT_ST[rmm]} segments")

# Step 1: Extract all segments into a structured dataset


In [ ]:
# Step 1: Extract all segments into a structured dataset
# Define main RMM classes to keep
main_RMM = {"hands flapping", "one hand flap", "jumping", "rocking", "spinning"}

# Initialize
segments = []
video_counter = {}
segment_global_id = 0

# Get column names
rmm_cols_names = [col for col in bw_df.columns if col.startswith('RMM_') and 'TS' not in col and "Notes" not in col and "Label" not in col]
rmm_cols_labels = [col for col in bw_df.columns if col.startswith('RMM_') and "Label" in col]
rmm_cols_ts = [col for col in bw_df.columns if col.startswith('RMM_') and "TS" in col]
rmm_cols = list(zip(rmm_cols_names, rmm_cols_labels, rmm_cols_ts))

skip_labels = {'Unsure', 'Debatable'}

# Iterate through all rows
for row_idx, row_data in bw_df.iterrows():
    # Extract video-level metadata
    child_id = row_data.get("ID", "unknown")
    timepoint = row_data.get("timepoint", "unknown")
    video_file = row_data.get("SourceFile", "")
    filename = row_data.get("FileName", "")
    
    # Handle missing timepoint
    if pd.isna(timepoint):
        timepoint = "unknown"
    else:
        timepoint = str(timepoint)
    
    # Create unique video ID
    video_key = (child_id, timepoint, filename)
    if video_key not in video_counter:
        video_counter[video_key] = len(video_counter)
    video_idx = video_counter[video_key]
    video_id = f"{child_id}_{timepoint}_{video_idx}"
    lcto_group = f"{child_id}_{timepoint}"
    
    # Iterate through RMM columns
    rmm_col_idx = 0
    for col_name, col_label, col_ts in rmm_cols:
        # Skip if no RMM type annotated
        if pd.isna(row_data[col_name]):
            continue
        
        # Skip if annotator marked as Unsure or Debatable
        if row_data[col_label] in skip_labels:
            continue
        
        # Get RMM type and standardize using bw_rmm_map
        rmm_type_raw = str(row_data[col_name]).strip()
        rmm_type = bw_rmm_map.get(rmm_type_raw, rmm_type_raw)
        
        # FILTER: Skip if not in main_RMM classes
        if rmm_type not in main_RMM:
            continue
        
        # Skip if no timestamps
        if pd.isna(row_data[col_ts]):
            continue
        
        # Parse timestamps
        ts_string = str(row_data[col_ts]).strip()
        parsed_timestamps = parse_timestamps(ts_string)
        
        # Create one entry per temporal segment
        for seg_idx, (start_sec, end_sec, quality) in enumerate(parsed_timestamps):
            segment_id = f"{child_id}_{timepoint}_{video_idx}_{rmm_col_idx}_{seg_idx}"
            
            segments.append({
                'segment_id': segment_id,
                'segment_global_id': segment_global_id,
                'video_id': video_id,
                'lcto_group': lcto_group,
                'child_id': child_id,
                'timepoint': timepoint,
                'video_file': video_file,
                'filename': filename,
                'rmm_type': rmm_type,
                'rmm_type_raw': rmm_type_raw,
                'start_sec': start_sec,
                'end_sec': end_sec,
                'duration': end_sec - start_sec,
                'quality_rating': quality,
                'annotator_label': row_data[col_label],
                'annotator': row_data.get('Original_Coder', 'unknown'),
                'video_duration': row_data.get('Duration_s', None),
                'n_children': row_data.get('#_children', None),
                'n_adults': row_data.get('#_adults', None),
            })
            
            segment_global_id += 1
        
        rmm_col_idx += 1

# Create DataFrame
segments_df = pd.DataFrame(segments)

print(f"Total segments extracted (main RMM classes only): {len(segments_df)}")
print(f"Total unique videos: {len(video_counter)}")
print(f"Total unique LCTO groups: {segments_df['lcto_group'].nunique()}")
print(f"\nSegments per RMM type (filtered to main classes):")
print(segments_df['rmm_type'].value_counts().sort_values(ascending=False))
print(f"\nDataFrame shape: {segments_df.shape}")
print(f"\nFirst few rows:")
segments_df.head()

In [ ]:
# Save to CSV
output_path = '/Users/brukewossenseged/Desktop/projects/sails/rmm_segments.csv'
segments_df.to_csv(output_path, index=False)
print(f"Saved segments to: {output_path}")

In [ ]:
# Generate comprehensive dataset summary

print("=" * 80)
print("DATASET SUMMARY REPORT")
print("=" * 80)

print("\n1. OVERALL STATISTICS")
print("-" * 80)
print(f"Total segments: {len(segments_df)}")
print(f"Total unique videos: {segments_df['video_id'].nunique()}")
print(f"Total unique children: {segments_df['child_id'].nunique()}")
print(f"Total unique LCTO groups (child, timepoint): {segments_df['lcto_group'].nunique()}")
print(f"Total unique timepoints: {segments_df['timepoint'].nunique()}")

print("\n2. RMM TYPE DISTRIBUTION")
print("-" * 80)
rmm_counts = segments_df['rmm_type'].value_counts().sort_values(ascending=False)
for rmm, count in rmm_counts.items():
    duration = segments_df[segments_df['rmm_type'] == rmm]['duration'].sum()
    print(f"{rmm:25s}: {count:4d} segments ({duration:7.1f} seconds total, {duration/count:5.2f} sec avg)")

print("\n3. TIMEPOINT DISTRIBUTION")
print("-" * 80)
tp_counts = segments_df['timepoint'].value_counts()
for tp, count in tp_counts.items():
    print(f"Timepoint {tp:10s}: {count:4d} segments")

print("\n4. LCTO GROUP STATISTICS")
print("-" * 80)
lcto_seg_counts = segments_df.groupby('lcto_group').size()
print(f"LCTO groups: {len(lcto_seg_counts)}")
print(f"Segments per LCTO group - Mean: {lcto_seg_counts.mean():.2f}, Median: {lcto_seg_counts.median():.1f}")
print(f"Segments per LCTO group - Min: {lcto_seg_counts.min()}, Max: {lcto_seg_counts.max()}")

print("\n5. DURATION STATISTICS")
print("-" * 80)
print(f"Total annotated duration: {segments_df['duration'].sum():.1f} seconds ({segments_df['duration'].sum()/60:.1f} minutes)")
print(f"Segment duration - Mean: {segments_df['duration'].mean():.2f} sec, Median: {segments_df['duration'].median():.2f} sec")
print(f"Segment duration - Min: {segments_df['duration'].min():.2f} sec, Max: {segments_df['duration'].max():.2f} sec")

print("\n6. QUALITY RATING DISTRIBUTION")
print("-" * 80)
quality_counts = segments_df['quality_rating'].value_counts(dropna=False).sort_index()
for qual, count in quality_counts.items():
    if pd.isna(qual):
        print(f"No quality rating: {count:4d} segments")
    else:
        print(f"Quality {qual}: {count:4d} segments")

print("\n7. ANNOTATOR LABEL DISTRIBUTION")
print("-" * 80)
label_counts = segments_df['annotator_label'].value_counts(dropna=False)
for label, count in label_counts.items():
    print(f"{str(label):25s}: {count:4d} segments")

print("\n8. ANNOTATOR DISTRIBUTION")
print("-" * 80)
annotator_counts = segments_df['annotator'].value_counts()
for ann, count in annotator_counts.items():
    print(f"{ann}: {count:4d} segments")

print("\n9. TOP LCTO GROUPS BY SEGMENT COUNT")
print("-" * 80)
top_lcto = segments_df.groupby('lcto_group').size().sort_values(ascending=False).head(10)
for group, count in top_lcto.items():
    print(f"{group:30s}: {count:3d} segments")

print("\n10. SEGMENTS PER VIDEO STATISTICS")
print("-" * 80)
segs_per_video = segments_df.groupby('video_id').size()
print(f"Videos: {len(segs_per_video)}")
print(f"Segments per video - Mean: {segs_per_video.mean():.2f}, Median: {segs_per_video.median():.1f}")
print(f"Segments per video - Min: {segs_per_video.min()}, Max: {segs_per_video.max()}")

print("\n" + "=" * 80)

In [ ]:
# Additional analysis: Check for potential issues

print("DATA QUALITY CHECKS")
print("=" * 80)

# Check for very short segments
short_segs = segments_df[segments_df['duration'] < 1.0]
print(f"\n1. Segments shorter than 1 second: {len(short_segs)}")
if len(short_segs) > 0:
    print("   Examples:")
    print(short_segs[['segment_id', 'rmm_type', 'duration']].head())

# Check for very long segments
long_segs = segments_df[segments_df['duration'] > 60.0]
print(f"\n2. Segments longer than 60 seconds: {len(long_segs)}")
if len(long_segs) > 0:
    print("   Examples:")
    print(long_segs[['segment_id', 'rmm_type', 'duration']].head())

# Check segments that are longer than video duration
invalid_segs = segments_df[segments_df['end_sec'] > segments_df['video_duration']]
print(f"\n3. Segments extending beyond video duration: {len(invalid_segs)}")
if len(invalid_segs) > 0:
    print("   Examples:")
    print(invalid_segs[['segment_id', 'end_sec', 'video_duration']].head())

# Check LCTO groups with very few segments
small_groups = segments_df.groupby('lcto_group').size()
small_groups = small_groups[small_groups < 3]
print(f"\n4. LCTO groups with fewer than 3 segments: {len(small_groups)}")
if len(small_groups) > 0:
    print("   These groups may be problematic for cross-validation:")
    for group, count in small_groups.items():
        print(f"   {group}: {count} segments")

# Check for class imbalance
print(f"\n5. Class imbalance ratio:")
max_class = segments_df['rmm_type'].value_counts().max()
min_class = segments_df['rmm_type'].value_counts().min()
print(f"   Most common class: {max_class} segments")
print(f"   Least common class: {min_class} segments")
print(f"   Imbalance ratio: {max_class/min_class:.1f}:1")

print("\n" + "=" * 80)

# Step 2: Generate LCTO (Leave-Child-Timepoint-Out) Cross-Validation Splits

This section creates K-fold cross-validation splits that respect the hierarchical structure of the data:
- All segments from the same (child, timepoint) pair stay together in train or validation
- Prevents data leakage between folds
- Attempts to maintain reasonable class distribution across folds

In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

# Configuration
N_FOLDS = 3  # Adjust this as needed

# Get unique LCTO groups
lcto_groups = segments_df['lcto_group'].unique()
print(f"Total LCTO groups: {len(lcto_groups)}")

# For GroupKFold, we need to create a mapping of segments to group indices
# Create a dictionary mapping lcto_group to a numeric index
group_to_idx = {group: idx for idx, group in enumerate(lcto_groups)}
segment_groups = segments_df['lcto_group'].map(group_to_idx).values

# Create GroupKFold splitter
gkf = GroupKFold(n_splits=N_FOLDS)

# Generate splits
splits = []
for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(segments_df, groups=segment_groups)):
    train_lcto_groups = set(segments_df.iloc[train_idx]['lcto_group'].unique())
    val_lcto_groups = set(segments_df.iloc[val_idx]['lcto_group'].unique())
    
    # Verify no overlap
    assert len(train_lcto_groups & val_lcto_groups) == 0, f"Fold {fold_idx}: Data leakage detected!"
    
    splits.append({
        'fold': fold_idx,
        'train_idx': train_idx,
        'val_idx': val_idx,
        'train_lcto_groups': train_lcto_groups,
        'val_lcto_groups': val_lcto_groups,
    })
    
    print(f"\nFold {fold_idx}:")
    print(f"  Train: {len(train_idx)} segments from {len(train_lcto_groups)} LCTO groups")
    print(f"  Val:   {len(val_idx)} segments from {len(val_lcto_groups)} LCTO groups")

print(f"\n✓ Generated {N_FOLDS} folds successfully")
print(f"✓ No data leakage detected (train/val LCTO groups are disjoint)")

In [ ]:
# Analyze class distribution across folds

print("\n" + "=" * 80)
print("CLASS DISTRIBUTION ACROSS FOLDS")
print("=" * 80)

for fold_idx, split in enumerate(splits):
    train_df = segments_df.iloc[split['train_idx']]
    val_df = segments_df.iloc[split['val_idx']]
    
    print(f"\nFold {fold_idx}:")
    print(f"{'RMM Type':<25s} {'Train':>8s} {'Val':>8s} {'Train %':>10s} {'Val %':>10s}")
    print("-" * 80)
    
    # Get all RMM types
    all_rmm_types = segments_df['rmm_type'].unique()
    
    for rmm in sorted(all_rmm_types):
        train_count = (train_df['rmm_type'] == rmm).sum()
        val_count = (val_df['rmm_type'] == rmm).sum()
        train_pct = 100 * train_count / len(train_df) if len(train_df) > 0 else 0
        val_pct = 100 * val_count / len(val_df) if len(val_df) > 0 else 0
        
        print(f"{rmm:<25s} {train_count:8d} {val_count:8d} {train_pct:9.1f}% {val_pct:9.1f}%")
    
    print("-" * 80)
    print(f"{'TOTAL':<25s} {len(train_df):8d} {len(val_df):8d} {100.0:9.1f}% {100.0:9.1f}%")

In [ ]:
# Save fold information and split files

import os
import json

# Create output directory for splits
splits_dir = '/Users/brukewossenseged/Desktop/projects/sails/cv_splits'
os.makedirs(splits_dir, exist_ok=True)

# Save overall split information
split_info = {
    'n_folds': N_FOLDS,
    'total_segments': len(segments_df),
    'total_lcto_groups': len(lcto_groups),
    'folds': []
}

# Save each fold
for fold_idx, split in enumerate(splits):
    train_df = segments_df.iloc[split['train_idx']]
    val_df = segments_df.iloc[split['val_idx']]
    
    # Save train and val CSVs
    train_path = os.path.join(splits_dir, f'fold_{fold_idx}_train.csv')
    val_path = os.path.join(splits_dir, f'fold_{fold_idx}_val.csv')
    
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    
    # Record fold information
    fold_info = {
        'fold': fold_idx,
        'train_segments': len(train_df),
        'val_segments': len(val_df),
        'train_lcto_groups': len(split['train_lcto_groups']),
        'val_lcto_groups': len(split['val_lcto_groups']),
        'train_file': train_path,
        'val_file': val_path,
        'train_lcto_group_list': sorted(list(split['train_lcto_groups'])),
        'val_lcto_group_list': sorted(list(split['val_lcto_groups'])),
    }
    split_info['folds'].append(fold_info)
    
    print(f"Saved Fold {fold_idx}:")
    print(f"  Train: {train_path}")
    print(f"  Val:   {val_path}")

# Save split metadata
metadata_path = os.path.join(splits_dir, 'split_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(split_info, f, indent=2)

print(f"\n✓ Saved all fold files to: {splits_dir}")
print(f"✓ Saved split metadata to: {metadata_path}")

## Alternative: Single Train/Val/Test Split

If you prefer a single split instead of K-fold CV, use this section:

In [ ]:
# Create a single train/val/test split respecting LCTO groups

from sklearn.model_selection import train_test_split

# Configuration
TRAIN_RATIO = 0.7
VAL_RATIO = 0.10
TEST_RATIO = 0.15

# Get all unique LCTO groups
all_lcto_groups = segments_df['lcto_group'].unique()
n_groups = len(all_lcto_groups)

# First split: train vs (val + test)
train_groups, valtest_groups = train_test_split(
    all_lcto_groups, 
    train_size=TRAIN_RATIO, 
    random_state=42
)

# Second split: val vs test
val_groups, test_groups = train_test_split(
    valtest_groups,
    train_size=VAL_RATIO/(VAL_RATIO + TEST_RATIO),
    random_state=42
)

# Create dataframes for each split
train_single_df = segments_df[segments_df['lcto_group'].isin(train_groups)]
val_single_df = segments_df[segments_df['lcto_group'].isin(val_groups)]
test_single_df = segments_df[segments_df['lcto_group'].isin(test_groups)]

# Verify no overlap
assert len(set(train_groups) & set(val_groups)) == 0, "Train/Val overlap detected!"
assert len(set(train_groups) & set(test_groups)) == 0, "Train/Test overlap detected!"
assert len(set(val_groups) & set(test_groups)) == 0, "Val/Test overlap detected!"

print("SINGLE TRAIN/VAL/TEST SPLIT")
print("=" * 80)
print(f"\nTrain: {len(train_single_df)} segments from {len(train_groups)} LCTO groups ({100*len(train_single_df)/len(segments_df):.1f}%)")
print(f"Val:   {len(val_single_df)} segments from {len(val_groups)} LCTO groups ({100*len(val_single_df)/len(segments_df):.1f}%)")
print(f"Test:  {len(test_single_df)} segments from {len(test_groups)} LCTO groups ({100*len(test_single_df)/len(segments_df):.1f}%)")

print("\nClass distribution:")
print(f"{'RMM Type':<25s} {'Train':>8s} {'Val':>8s} {'Test':>8s}")
print("-" * 80)

for rmm in sorted(segments_df['rmm_type'].unique()):
    train_count = (train_single_df['rmm_type'] == rmm).sum()
    val_count = (val_single_df['rmm_type'] == rmm).sum()
    test_count = (test_single_df['rmm_type'] == rmm).sum()
    print(f"{rmm:<25s} {train_count:8d} {val_count:8d} {test_count:8d}")

print("=" * 80)

# Save the single split
single_split_dir = '/Users/brukewossenseged/Desktop/projects/sails/single_split'
os.makedirs(single_split_dir, exist_ok=True)

train_single_df.to_csv(os.path.join(single_split_dir, 'train.csv'), index=False)
val_single_df.to_csv(os.path.join(single_split_dir, 'val.csv'), index=False)
test_single_df.to_csv(os.path.join(single_split_dir, 'test.csv'), index=False)

# Save metadata
single_split_info = {
    'train_segments': len(train_single_df),
    'val_segments': len(val_single_df),
    'test_segments': len(test_single_df),
    'train_lcto_groups': len(train_groups),
    'val_lcto_groups': len(val_groups),
    'test_lcto_groups': len(test_groups),
    'train_lcto_group_list': sorted(list(train_groups)),
    'val_lcto_group_list': sorted(list(val_groups)),
    'test_lcto_group_list': sorted(list(test_groups)),
}

with open(os.path.join(single_split_dir, 'split_info.json'), 'w') as f:
    json.dump(single_split_info, f, indent=2)

print(f"\n✓ Saved single split to: {single_split_dir}")

# Step 1b: Create Combined Dataset (hands flapping + one hand flap merged)

This section creates an alternative dataset where "hands flapping" and "one hand flap" are combined into a single "hands flapping" class, reducing from 5 to 4 classes.


In [ ]:
# Create combined dataset with hands flapping + one hand flap merged
# Define main RMM classes (4 classes instead of 5)
main_RMM_combined = {"hands flapping", "jumping", "rocking", "spinning"}

# Extended mapping that combines hand flapping variants into "hands flapping"
bw_rmm_map_combined = {
    'arm flapping': 'hands flapping',
    'arms flapping': 'hands flapping',
    'one arm flap': 'hands flapping',
    'one arm flapping': 'hands flapping',
    'one hand flap': 'hands flapping',
    'bouncing': 'jumping',
    'twisting': 'rocking',
    'bed jumping': 'jumping',
    'knee jumping': 'jumping',
}

# Initialize
segments_combined = []
video_counter_combined = {}
segment_global_id_combined = 0

# Get column names
rmm_cols_names = [col for col in bw_df.columns if col.startswith('RMM_') and 'TS' not in col and "Notes" not in col and "Label" not in col]
rmm_cols_labels = [col for col in bw_df.columns if col.startswith('RMM_') and "Label" in col]
rmm_cols_ts = [col for col in bw_df.columns if col.startswith('RMM_') and "TS" in col]
rmm_cols_combined = list(zip(rmm_cols_names, rmm_cols_labels, rmm_cols_ts))

skip_labels = {'Unsure', 'Debatable'}

# Iterate through all rows
for row_idx, row_data in bw_df.iterrows():
    # Extract video-level metadata
    child_id = row_data.get("ID", "unknown")
    timepoint = row_data.get("timepoint", "unknown")
    video_file = row_data.get("SourceFile", "")
    filename = row_data.get("FileName", "")
    
    # Handle missing timepoint
    if pd.isna(timepoint):
        timepoint = "unknown"
    else:
        timepoint = str(timepoint)
    
    # Create unique video ID
    video_key = (child_id, timepoint, filename)
    if video_key not in video_counter_combined:
        video_counter_combined[video_key] = len(video_counter_combined)
    video_idx = video_counter_combined[video_key]
    video_id = f"{child_id}_{timepoint}_{video_idx}"
    lcto_group = f"{child_id}_{timepoint}"
    
    # Iterate through RMM columns
    rmm_col_idx = 0
    for col_name, col_label, col_ts in rmm_cols_combined:
        # Skip if no RMM type annotated
        if pd.isna(row_data[col_name]):
            continue
        
        # Skip if annotator marked as Unsure or Debatable
        if row_data[col_label] in skip_labels:
            continue
        
        # Get RMM type and standardize using combined mapping
        rmm_type_raw = str(row_data[col_name]).strip()
        rmm_type = bw_rmm_map_combined.get(rmm_type_raw, rmm_type_raw)
        
        # FILTER: Skip if not in main_RMM_combined classes
        if rmm_type not in main_RMM_combined:
            continue
        
        # Skip if no timestamps
        if pd.isna(row_data[col_ts]):
            continue
        
        # Parse timestamps
        ts_string = str(row_data[col_ts]).strip()
        parsed_timestamps = parse_timestamps(ts_string)
        
        # Create one entry per temporal segment
        for seg_idx, (start_sec, end_sec, quality) in enumerate(parsed_timestamps):
            segment_id = f"{child_id}_{timepoint}_{video_idx}_{rmm_col_idx}_{seg_idx}"
            
            segments_combined.append({
                'segment_id': segment_id,
                'segment_global_id': segment_global_id_combined,
                'video_id': video_id,
                'lcto_group': lcto_group,
                'child_id': child_id,
                'timepoint': timepoint,
                'video_file': video_file,
                'filename': filename,
                'rmm_type': rmm_type,
                'rmm_type_raw': rmm_type_raw,
                'start_sec': start_sec,
                'end_sec': end_sec,
                'duration': end_sec - start_sec,
                'quality_rating': quality,
                'annotator_label': row_data[col_label],
                'annotator': row_data.get('Original_Coder', 'unknown'),
                'video_duration': row_data.get('Duration_s', None),
                'n_children': row_data.get('#_children', None),
                'n_adults': row_data.get('#_adults', None),
            })
            
            segment_global_id_combined += 1
        
        rmm_col_idx += 1

# Create DataFrame
segments_combined_df = pd.DataFrame(segments_combined)

print(f"Total segments extracted (4-class combined): {len(segments_combined_df)}")
print(f"Total unique videos: {len(video_counter_combined)}")
print(f"Total unique LCTO groups: {segments_combined_df['lcto_group'].nunique()}")
print(f"\nSegments per RMM type (combined classes):")
print(segments_combined_df['rmm_type'].value_counts().sort_values(ascending=False))
print(f"\nDataFrame shape: {segments_combined_df.shape}")
print(f"\nFirst few rows:")
segments_combined_df.head()


In [ ]:
# Save combined segments to CSV
output_path_combined = '/Users/brukewossenseged/Desktop/projects/sails/rmm_segments_4class.csv'
segments_combined_df.to_csv(output_path_combined, index=False)
print(f"Saved combined segments to: {output_path_combined}")

# Summary comparison
print("\n" + "=" * 80)
print("COMPARISON: 5-CLASS vs 4-CLASS DATASETS")
print("=" * 80)
print(f"\n5-class dataset: {len(segments_df)} segments")
print(segments_df['rmm_type'].value_counts())
print(f"\n4-class dataset (combined): {len(segments_combined_df)} segments")
print(segments_combined_df['rmm_type'].value_counts())


# Step 2b: Generate LCTO CV Splits for 4-Class Combined Dataset

Creates K-fold cross-validation splits for the combined (4-class) dataset:
- Classes: hands flapping (combined), jumping, rocking, spinning
- All segments from same (child, timepoint) pair stay together


In [ ]:
# Generate K-Fold CV splits for 4-class combined dataset
from sklearn.model_selection import GroupKFold
import numpy as np

# Configuration
N_FOLDS_COMBINED = 3

# Get unique LCTO groups from combined dataset
lcto_groups_combined = segments_combined_df['lcto_group'].unique()
print(f"Total LCTO groups (4-class): {len(lcto_groups_combined)}")

# Create mapping of segments to group indices
group_to_idx_combined = {group: idx for idx, group in enumerate(lcto_groups_combined)}
segment_groups_combined = segments_combined_df['lcto_group'].map(group_to_idx_combined).values

# Create GroupKFold splitter
gkf_combined = GroupKFold(n_splits=N_FOLDS_COMBINED)

# Generate splits
splits_combined = []
for fold_idx, (train_idx, val_idx) in enumerate(gkf_combined.split(segments_combined_df, groups=segment_groups_combined)):
    train_lcto_groups = set(segments_combined_df.iloc[train_idx]['lcto_group'].unique())
    val_lcto_groups = set(segments_combined_df.iloc[val_idx]['lcto_group'].unique())
    
    # Verify no overlap
    assert len(train_lcto_groups & val_lcto_groups) == 0, f"Fold {fold_idx}: Data leakage detected!"
    
    splits_combined.append({
        'fold': fold_idx,
        'train_idx': train_idx,
        'val_idx': val_idx,
        'train_lcto_groups': train_lcto_groups,
        'val_lcto_groups': val_lcto_groups,
    })
    
    print(f"\nFold {fold_idx}:")
    print(f"  Train: {len(train_idx)} segments from {len(train_lcto_groups)} LCTO groups")
    print(f"  Val:   {len(val_idx)} segments from {len(val_lcto_groups)} LCTO groups")

print(f"\n✓ Generated {N_FOLDS_COMBINED} folds for 4-class dataset successfully")
print(f"✓ No data leakage detected")


In [ ]:
# Analyze class distribution across folds for 4-class dataset
print("\n" + "=" * 80)
print("CLASS DISTRIBUTION ACROSS FOLDS (4-CLASS COMBINED)")
print("=" * 80)

for fold_idx, split in enumerate(splits_combined):
    train_df = segments_combined_df.iloc[split['train_idx']]
    val_df = segments_combined_df.iloc[split['val_idx']]
    
    print(f"\nFold {fold_idx}:")
    print(f"{'RMM Type':<25s} {'Train':>8s} {'Val':>8s} {'Train %':>10s} {'Val %':>10s}")
    print("-" * 80)
    
    # Get all RMM types
    all_rmm_types = segments_combined_df['rmm_type'].unique()
    
    for rmm in sorted(all_rmm_types):
        train_count = (train_df['rmm_type'] == rmm).sum()
        val_count = (val_df['rmm_type'] == rmm).sum()
        train_pct = 100 * train_count / len(train_df) if len(train_df) > 0 else 0
        val_pct = 100 * val_count / len(val_df) if len(val_df) > 0 else 0
        
        print(f"{rmm:<25s} {train_count:8d} {val_count:8d} {train_pct:9.1f}% {val_pct:9.1f}%")
    
    print("-" * 80)
    print(f"{'TOTAL':<25s} {len(train_df):8d} {len(val_df):8d} {100.0:9.1f}% {100.0:9.1f}%")


In [ ]:
# Save fold information and split files for 4-class dataset
import os
import json

# Create output directory for 4-class splits
splits_dir_combined = '/Users/brukewossenseged/Desktop/projects/sails/cv_splits_4class'
os.makedirs(splits_dir_combined, exist_ok=True)

# Save overall split information
split_info_combined = {
    'n_folds': N_FOLDS_COMBINED,
    'n_classes': 4,
    'classes': ['hands flapping', 'jumping', 'rocking', 'spinning'],
    'total_segments': len(segments_combined_df),
    'total_lcto_groups': len(lcto_groups_combined),
    'folds': []
}

# Save each fold
for fold_idx, split in enumerate(splits_combined):
    train_df = segments_combined_df.iloc[split['train_idx']]
    val_df = segments_combined_df.iloc[split['val_idx']]
    
    # Save train and val CSVs
    train_path = os.path.join(splits_dir_combined, f'fold_{fold_idx}_train.csv')
    val_path = os.path.join(splits_dir_combined, f'fold_{fold_idx}_val.csv')
    
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    
    # Record fold information
    fold_info = {
        'fold': fold_idx,
        'train_segments': len(train_df),
        'val_segments': len(val_df),
        'train_lcto_groups': len(split['train_lcto_groups']),
        'val_lcto_groups': len(split['val_lcto_groups']),
        'train_file': train_path,
        'val_file': val_path,
        'train_lcto_group_list': sorted(list(split['train_lcto_groups'])),
        'val_lcto_group_list': sorted(list(split['val_lcto_groups'])),
    }
    split_info_combined['folds'].append(fold_info)
    
    print(f"Saved Fold {fold_idx}:")
    print(f"  Train: {train_path}")
    print(f"  Val:   {val_path}")

# Save split metadata
metadata_path_combined = os.path.join(splits_dir_combined, 'split_metadata.json')
with open(metadata_path_combined, 'w') as f:
    json.dump(split_info_combined, f, indent=2)

print(f"\n✓ Saved all 4-class fold files to: {splits_dir_combined}")
print(f"✓ Saved split metadata to: {metadata_path_combined}")


In [ ]:
# Create single train/val/test split for 4-class dataset
from sklearn.model_selection import train_test_split

# Configuration (same ratios as 5-class)
TRAIN_RATIO = 0.7
VAL_RATIO = 0.10
TEST_RATIO = 0.15

# Get all unique LCTO groups from combined dataset
all_lcto_groups_combined = segments_combined_df['lcto_group'].unique()

# First split: train vs (val + test)
train_groups_c, valtest_groups_c = train_test_split(
    all_lcto_groups_combined, 
    train_size=TRAIN_RATIO, 
    random_state=42
)

# Second split: val vs test
val_groups_c, test_groups_c = train_test_split(
    valtest_groups_c,
    train_size=VAL_RATIO/(VAL_RATIO + TEST_RATIO),
    random_state=42
)

# Create dataframes for each split
train_single_combined_df = segments_combined_df[segments_combined_df['lcto_group'].isin(train_groups_c)]
val_single_combined_df = segments_combined_df[segments_combined_df['lcto_group'].isin(val_groups_c)]
test_single_combined_df = segments_combined_df[segments_combined_df['lcto_group'].isin(test_groups_c)]

# Verify no overlap
assert len(set(train_groups_c) & set(val_groups_c)) == 0, "Train/Val overlap detected!"
assert len(set(train_groups_c) & set(test_groups_c)) == 0, "Train/Test overlap detected!"
assert len(set(val_groups_c) & set(test_groups_c)) == 0, "Val/Test overlap detected!"

print("SINGLE TRAIN/VAL/TEST SPLIT (4-CLASS COMBINED)")
print("=" * 80)
print(f"\nTrain: {len(train_single_combined_df)} segments from {len(train_groups_c)} LCTO groups ({100*len(train_single_combined_df)/len(segments_combined_df):.1f}%)")
print(f"Val:   {len(val_single_combined_df)} segments from {len(val_groups_c)} LCTO groups ({100*len(val_single_combined_df)/len(segments_combined_df):.1f}%)")
print(f"Test:  {len(test_single_combined_df)} segments from {len(test_groups_c)} LCTO groups ({100*len(test_single_combined_df)/len(segments_combined_df):.1f}%)")

print("\nClass distribution:")
print(f"{'RMM Type':<25s} {'Train':>8s} {'Val':>8s} {'Test':>8s}")
print("-" * 80)

for rmm in sorted(segments_combined_df['rmm_type'].unique()):
    train_count = (train_single_combined_df['rmm_type'] == rmm).sum()
    val_count = (val_single_combined_df['rmm_type'] == rmm).sum()
    test_count = (test_single_combined_df['rmm_type'] == rmm).sum()
    print(f"{rmm:<25s} {train_count:8d} {val_count:8d} {test_count:8d}")

print("=" * 80)

# Save the single split for 4-class
single_split_dir_combined = '/Users/brukewossenseged/Desktop/projects/sails/single_split_4class'
os.makedirs(single_split_dir_combined, exist_ok=True)

train_single_combined_df.to_csv(os.path.join(single_split_dir_combined, 'train.csv'), index=False)
val_single_combined_df.to_csv(os.path.join(single_split_dir_combined, 'val.csv'), index=False)
test_single_combined_df.to_csv(os.path.join(single_split_dir_combined, 'test.csv'), index=False)

# Save metadata
single_split_info_combined = {
    'n_classes': 4,
    'classes': ['hands flapping', 'jumping', 'rocking', 'spinning'],
    'train_segments': len(train_single_combined_df),
    'val_segments': len(val_single_combined_df),
    'test_segments': len(test_single_combined_df),
    'train_lcto_groups': len(train_groups_c),
    'val_lcto_groups': len(val_groups_c),
    'test_lcto_groups': len(test_groups_c),
    'train_lcto_group_list': sorted(list(train_groups_c)),
    'val_lcto_group_list': sorted(list(val_groups_c)),
    'test_lcto_group_list': sorted(list(test_groups_c)),
}

with open(os.path.join(single_split_dir_combined, 'split_info.json'), 'w') as f:
    json.dump(single_split_info_combined, f, indent=2)

print(f"\n✓ Saved 4-class single split to: {single_split_dir_combined}")


In [ ]:
# Summary of Step 2

print("\n" + "=" * 80)
print("STEP 2 COMPLETE: LCTO CV SPLITS GENERATED")
print("=" * 80)

print("\n✓ K-Fold Cross-Validation:")
print(f"  - {N_FOLDS} folds created")
print(f"  - All folds respect LCTO grouping (no data leakage)")
print(f"  - Saved to: {splits_dir}")
print(f"  - Metadata: {metadata_path}")

print("\n✓ Single Train/Val/Test Split:")
print(f"  - 70/15/15 split created")
print(f"  - Respects LCTO grouping (no data leakage)")
print(f"  - Saved to: {single_split_dir}")

print("\n✓ Data Integrity Verified:")
print(f"  - No overlap between train/val LCTO groups in any fold")
print(f"  - All segments from same (child, timepoint) stay together")

print("\n📊 Next Steps:")
print("  - Review class distribution across folds")
print("  - Choose between K-fold CV or single split based on your needs")
print("  - Proceed to Step 3: Generate TAL format (optional)")
print("  - Start training your models!")

print("=" * 80)